# Bernoulli MLE and the Log-Loss Function

## Introduction

This notebook derives how the **Bernoulli probability mass function (PMF)** leads to the **Negative Log-Likelihood (Log-Loss)** cost function used in binary classification. We show that minimizing log-loss is equivalent to maximizing the likelihood of observed data under the assumption that labels follow a Bernoulli distribution.

---

## The Bernoulli Distribution

For a binary random variable $Y \in \{0, 1\}$ with parameter $p \in [0, 1]$, the Bernoulli PMF is:

$$P(Y = y \mid p) = p^y (1-p)^{1-y}$$

**Interpretation:**
- When $y = 1$: $P(Y=1 \mid p) = p$
- When $y = 0$: $P(Y=0 \mid p) = 1-p$

This compact notation elegantly handles both cases.

---

## Logistic Regression Setup

In binary classification with features $x_i \in \mathbb{R}^n$ and labels $y_i \in \{0, 1\}$, we model:

$$P(y_i = 1 \mid x_i; \theta) = \sigma(\theta^T x_i) = \frac{1}{1 + e^{-\theta^T x_i}}$$

where $\sigma(z)$ is the **sigmoid function**.

**Key Property:** The sigmoid maps any real number to $(0, 1)$, making it a valid probability.

Let $p_i = \sigma(\theta^T x_i)$ be the predicted probability for sample $i$.

---

## Likelihood Function

For a dataset of $m$ independent samples $(x_1, y_1), \ldots, (x_m, y_m)$, the **likelihood** of the parameters $\theta$ is:

$$\mathcal{L}(\theta) = \prod_{i=1}^{m} P(y_i \mid x_i; \theta)$$

Substituting the Bernoulli PMF:

$$\mathcal{L}(\theta) = \prod_{i=1}^{m} p_i^{y_i} (1-p_i)^{1-y_i}$$

**Interpretation:** This is the probability of observing the entire dataset given parameters $\theta$.

**Goal of MLE:** Find $\theta$ that maximizes $\mathcal{L}(\theta)$.

---

## Log-Likelihood Derivation

Products are difficult to optimize. We take the **natural logarithm**, which is monotonically increasing (preserves the maximizer):

$$\ell(\theta) = \log \mathcal{L}(\theta) = \log \prod_{i=1}^{m} p_i^{y_i} (1-p_i)^{1-y_i}$$

**Step 1:** Apply log to product (becomes sum):

$$\ell(\theta) = \sum_{i=1}^{m} \log \left[ p_i^{y_i} (1-p_i)^{1-y_i} \right]$$

**Step 2:** Use log property $\log(ab) = \log a + \log b$:

$$\ell(\theta) = \sum_{i=1}^{m} \left[ y_i \log p_i + (1-y_i) \log(1-p_i) \right]$$

**Step 3:** Divide by $m$ (average log-likelihood):

$$\ell(\theta) = \frac{1}{m} \sum_{i=1}^{m} \left[ y_i \log p_i + (1-y_i) \log(1-p_i) \right]$$

---

## From Maximization to Minimization

Machine learning conventionally frames optimization as **minimization**. We define the **Negative Log-Likelihood (NLL)** as the **cost function**:

$$J(\theta) = -\ell(\theta) = -\frac{1}{m} \sum_{i=1}^{m} \left[ y_i \log p_i + (1-y_i) \log(1-p_i) \right]$$

**Key Insight:**

$$\boxed{\text{Maximizing } \ell(\theta) \iff \text{Minimizing } J(\theta)}$$

This $J(\theta)$ is the **Binary Cross-Entropy Loss** (also called **Log-Loss**).

---

## Expanded Form

Substituting $p_i = \sigma(\theta^T x_i)$:

$$J(\theta) = -\frac{1}{m} \sum_{i=1}^{m} \left[ y_i \log \sigma(\theta^T x_i) + (1-y_i) \log(1 - \sigma(\theta^T x_i)) \right]$$

**This is the cost function we minimize in logistic regression.**

---

## Why This Loss Function?

The log-loss is **not arbitrary**. It emerges naturally from:

1. **Probabilistic Modeling:** Assuming labels follow a Bernoulli distribution
2. **Maximum Likelihood Estimation:** Finding parameters that best explain observed data
3. **Convexity:** The resulting cost function is convex, guaranteeing a global minimum

**Alternative Perspective:** Log-loss measures the **Kullback-Leibler divergence** between the true distribution and predicted distribution.

---

## Gradient of the Cost Function

To optimize via gradient descent, we need $\nabla J(\theta)$.

**Derivation:**

Starting from:
$$J(\theta) = -\frac{1}{m} \sum_{i=1}^{m} \left[ y_i \log p_i + (1-y_i) \log(1-p_i) \right]$$

**Step 1:** Derivative with respect to $p_i$:

$$\frac{\partial J}{\partial p_i} = -\frac{1}{m} \left[ \frac{y_i}{p_i} - \frac{1-y_i}{1-p_i} \right]$$

**Step 2:** Simplify:

$$\frac{\partial J}{\partial p_i} = -\frac{1}{m} \cdot \frac{y_i(1-p_i) - (1-y_i)p_i}{p_i(1-p_i)} = -\frac{1}{m} \cdot \frac{y_i - p_i}{p_i(1-p_i)}$$

**Step 3:** Chain rule with $p_i = \sigma(z_i)$ where $z_i = \theta^T x_i$:

Recall: $\frac{d\sigma(z)}{dz} = \sigma(z)(1 - \sigma(z))$

$$\frac{\partial p_i}{\partial \theta} = \sigma(z_i)(1-\sigma(z_i)) \cdot x_i = p_i(1-p_i) \cdot x_i$$

**Step 4:** Combine via chain rule:

$$\frac{\partial J}{\partial \theta} = \frac{\partial J}{\partial p_i} \cdot \frac{\partial p_i}{\partial \theta} = -\frac{1}{m} \cdot \frac{y_i - p_i}{p_i(1-p_i)} \cdot p_i(1-p_i) \cdot x_i$$

**Step 5:** Cancel terms:

$$\frac{\partial J}{\partial \theta} = -\frac{1}{m}(y_i - p_i) x_i$$

**Step 6:** Sum over all samples:

$$\nabla J(\theta) = \frac{1}{m} \sum_{i=1}^{m} (p_i - y_i) x_i$$

**Vectorized form:**

$$\boxed{\nabla J(\theta) = \frac{1}{m} X^T (\sigma(X\theta) - y)}$$

where $X \in \mathbb{R}^{m \times n}$ is the design matrix.

**Remarkable simplicity:** The gradient has the same form as linear regression!

---

## Numerical Stability

Direct computation of $\log(\sigma(z))$ can cause numerical issues when $z$ is very large or small.

**Stable formulation:**

$$\log(\sigma(z)) = \log\left(\frac{1}{1+e^{-z}}\right) = -\log(1 + e^{-z})$$

**For positive $z$:**
$$-\log(1 + e^{-z})$$

**For negative $z$:**
$$z - \log(1 + e^{z})$$

This avoids overflow/underflow in the exponential.

---

## Connection to Information Theory

The log-loss measures the **cross-entropy** between:
- True distribution: $q(y) = y$ (labels are deterministic)
- Predicted distribution: $p(y) = \sigma(\theta^T x)$

**Cross-entropy:** $H(q, p) = -\mathbb{E}_q[\log p]$

For binary classification:
$$H(y, p) = -[y \log p + (1-y) \log(1-p)]$$

Averaged over dataset → Log-Loss.

---

## Summary

| Concept | Formula |
|---------|---------|
| **Bernoulli PMF** | $P(y \mid p) = p^y (1-p)^{1-y}$ |
| **Likelihood** | $\mathcal{L}(\theta) = \prod_{i=1}^{m} p_i^{y_i} (1-p_i)^{1-y_i}$ |
| **Log-Likelihood** | $\ell(\theta) = \frac{1}{m} \sum_{i=1}^{m} [y_i \log p_i + (1-y_i) \log(1-p_i)]$ |
| **Cost Function (Log-Loss)** | $J(\theta) = -\ell(\theta)$ |
| **Gradient** | $\nabla J(\theta) = \frac{1}{m} X^T (p - y)$ |

**Key Takeaway:** Log-loss is not an arbitrary choice—it's the statistically optimal cost function when labels follow a Bernoulli distribution.
